# 04 — Feature Engineering & Model Training

Build ML-ready features and train scikit-learn models using a chronological time-series split.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

files = list(Path("../data").glob("*.csv")) or list(Path("data").glob("*.csv"))
if not files:
    raise FileNotFoundError("Place the Reliance OHLCV CSV in the project's data/ folder.")
df = pd.read_csv(files[0])
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for c in ["Open","High","Low","Close","Volume"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Date","Open","High","Low","Close"]).drop_duplicates().sort_values("Date").reset_index(drop=True)
df.head()


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2000-01-03,RELIANCE,EQ,233.05,237.50,251.70,237.50,251.70,251.70,249.37,4456424,1.111319e+14,NaN,NaN,NaN
1,2000-01-04,RELIANCE,EQ,251.70,258.40,271.85,251.30,271.85,271.85,263.52,9487878,2.500222e+14,NaN,NaN,NaN
2,2000-01-05,RELIANCE,EQ,271.85,256.65,287.90,256.65,286.75,282.50,274.79,26833684,7.373697e+14,NaN,NaN,NaN
3,2000-01-06,RELIANCE,EQ,282.50,289.00,300.70,289.00,293.50,294.35,295.45,15682286,4.633254e+14,NaN,NaN,NaN
4,2000-01-07,RELIANCE,EQ,294.35,295.00,317.90,293.00,314.50,314.55,308.91,19870977,6.138388e+14,NaN,NaN,NaN


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression

p=df["Close"]
f=pd.DataFrame(index=df.index)
f["Return_1D"]=p.pct_change()
f["Return_5D"]=p.pct_change(5)
f["Return_10D"]=p.pct_change(10)
f["SMA5_Ratio"]=p/p.rolling(5).mean()-1
f["SMA20_Ratio"]=p/p.rolling(20).mean()-1
f["Volatility_5D"]=f["Return_1D"].rolling(5).std()
f["Volatility_20D"]=f["Return_1D"].rolling(20).std()
f["Daily_Range"]=(df["High"]-df["Low"])/df["Close"]
f["Volume_Change"]=df["Volume"].pct_change()

delta=p.diff()
gain=delta.clip(lower=0).rolling(14).mean()
loss=(-delta.clip(upper=0)).rolling(14).mean()
f["RSI"]=100-100/(1+gain/loss)

target_price=p.shift(-1)
target_direction=(target_price>p).astype(int)
data=pd.concat([f,target_price.rename("Next_Close"),target_direction.rename("Direction"),p.rename("Close")],axis=1).dropna()
features=f.columns.tolist()
cut=int(len(data)*.8)
train,test=data.iloc[:cut],data.iloc[cut:]
print("Features:",features)
print("Train:",len(train),"Test:",len(test))

Features: ['Return_1D', 'Return_5D', 'Return_10D', 'SMA5_Ratio', 'SMA20_Ratio', 'Volatility_5D', 'Volatility_20D', 'Daily_Range', 'Volume_Change', 'RSI']
Train: 4228 Test: 1057


In [5]:
price_model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",Ridge(alpha=1.0))])
direction_model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(max_iter=2000))])
price_model.fit(train[features],train["Next_Close"])
direction_model.fit(train[features],train["Direction"])
price_pred=price_model.predict(test[features])
direction_pred=direction_model.predict(test[features])
print("Training complete.")

Training complete.
